# Disaster Risk Projection Data Analysis

This notebook analyzes disaster risk projection data for the northern hemisphere, focusing on the US, Canada, and Mexico.
It uses FEMA's National Risk Index and other international data sources to create a comprehensive risk assessment.

## Data Sources

- **FEMA National Risk Index (NRI)**: Provides risk scores for 18 natural hazards across US counties
- **FEMA OpenFEMA API**: Disaster declarations and related data
- **RAPT (Resilience Analysis and Planning Tool)**: Hazard mapping and risk analysis

## Output

The notebook generates **per-state CSV files** (e.g., `california_risk_data.csv`, `texas_risk_data.csv`) with:
- Rows: County FIPS codes within that state
- Columns: Various hazard risk scores mapped to RAPT hazard codes

This per-state approach reduces data lookup costs and makes the data more manageable for state-level analysis.

## Setup and Dependencies

First, let's install and import all necessary libraries.

In [ ]:
# If you need to install additional packages, uncomment and run:
# !uv add requests pandas numpy matplotlib seaborn tqdm

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Create output directory
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Setup complete. Output will be saved to: {OUTPUT_DIR.absolute()}")

## 1. FEMA National Risk Index Data

The National Risk Index (NRI) provides risk scores for 18 natural hazards at the county level.
We'll fetch this data from FEMA's OpenFEMA API.

In [ ]:
# FEMA OpenFEMA API endpoints
# Note: The FEMA API may have CORS restrictions when called from browsers.
# This notebook uses direct HTTP requests which bypass CORS issues.
NRI_API_BASE = "https://www.fema.gov/api/open/v2/FimaNriNationalRiskIndex"

# Alternative: Download full CSV from FEMA if API is unavailable
# NRI_CSV_URL = "https://hazards.fema.gov/nri/Content/StaticDocuments/DataDownload/NRI_Table_Counties/NRI_Table_Counties.csv"

# RAPT Hazard Type Codes (based on FEMA RAPT and risk data library)
# Mapping FEMA NRI hazards to standardized hazard codes
HAZARD_MAPPING = {
    'AVLN': 'Avalanche',
    'CFLD': 'Coastal Flooding',
    'CWAV': 'Cold Wave',
    'DRGT': 'Drought',
    'ERQK': 'Earthquake',
    'HAIL': 'Hail',
    'HWAV': 'Heat Wave',
    'HRCN': 'Hurricane',
    'ISTM': 'Ice Storm',
    'LNDS': 'Landslide',
    'LTNG': 'Lightning',
    'RFLD': 'Riverine Flooding',
    'SWND': 'Strong Wind',
    'TRND': 'Tornado',
    'TSUN': 'Tsunami',
    'VLCN': 'Volcanic Activity',
    'WFIR': 'Wildfire',
    'WNTW': 'Winter Weather'
}

print(f"Hazard types to analyze: {len(HAZARD_MAPPING)}")
for code, name in HAZARD_MAPPING.items():
    print(f"  {code}: {name}")

In [ ]:
def fetch_nri_data(limit=1000, offset=0):
    """
    Fetch National Risk Index data from FEMA API.
    
    CORS Note: When running in Jupyter notebooks, Python requests bypass browser CORS restrictions.
    This allows direct access to the FEMA API without CORS issues.
    
    Args:
        limit: Number of records to fetch per request (max 1000)
        offset: Starting offset for pagination
    
    Returns:
        DataFrame with NRI data
    """
    params = {
        '$top': limit,
        '$skip': offset,
        '$format': 'json'
    }
    
    try:
        # Using requests library bypasses browser CORS restrictions
        response = requests.get(NRI_API_BASE, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        if 'FimaNriNationalRiskIndex' in data:
            return pd.DataFrame(data['FimaNriNationalRiskIndex'])
        else:
            print(f"Unexpected response format: {list(data.keys())}")
            return pd.DataFrame()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching NRI data: {e}")
        return pd.DataFrame()

def fetch_all_nri_data():
    """
    Fetch all NRI data using pagination.
    
    Returns:
        DataFrame with all NRI records
    """
    all_data = []
    offset = 0
    limit = 1000
    
    print("Fetching FEMA National Risk Index data from live API...")
    print("Note: Python requests bypass browser CORS restrictions.")
    
    with tqdm(desc="Fetching NRI data") as pbar:
        while True:
            df_batch = fetch_nri_data(limit=limit, offset=offset)
            
            if df_batch.empty:
                break
            
            all_data.append(df_batch)
            pbar.update(len(df_batch))
            
            if len(df_batch) < limit:
                break
            
            offset += limit
    
    if all_data:
        df = pd.concat(all_data, ignore_index=True)
        print(f"\n✓ Fetched {len(df):,} total records from FEMA API")
        return df
    else:
        print("No data fetched from API")
        return pd.DataFrame()

# Note: The next cell will attempt to fetch live data and fall back to sample data if needed

In [ ]:
# Attempt to fetch from live FEMA API
# This bypasses CORS issues since we're using Python requests, not browser XHR
print("Attempting to fetch live data from FEMA API...")
print("This may take several minutes depending on API response time.\n")

try:
    nri_df = fetch_all_nri_data()
    
    if nri_df.empty:
        raise Exception("No data returned from API")
    
    print("\n" + "="*60)
    print("✓ SUCCESS: Using live FEMA API data")
    print("="*60)
    
except Exception as e:
    print(f"\n⚠ Could not fetch from API: {e}")
    print("\nFalling back to sample data...")
    
    sample_file = Path('sample_data/sample_nri_data.csv')
    if sample_file.exists():
        nri_df = pd.read_csv(sample_file)
        print(f"✓ Loaded {len(nri_df)} records from sample data")
        print("\n" + "="*60)
        print("Note: Using SAMPLE data (not live API data)")
        print("="*60)
        print("\nTo use real data, either:")
        print("  1. Ensure network access to www.fema.gov is available")
        print("  2. Download NRI data from https://hazards.fema.gov/nri/data-resources#csvDownload")
        print("  3. Load the downloaded CSV directly with: nri_df = pd.read_csv('path/to/NRI_Table_Counties.csv')")
    else:
        print(f"Error: Sample data file not found at {sample_file.absolute()}")
        print("Run 'python generate_sample_data.py' to create sample data")
        nri_df = pd.DataFrame()

In [ ]:
# Inspect the data structure
if not nri_df.empty:
    print(f"Shape: {nri_df.shape}")
    print(f"\nColumns: {list(nri_df.columns)}")
    print(f"\nFirst few rows:")
    display(nri_df.head())
    print(f"\nData types:")
    print(nri_df.dtypes)
else:
    print("No data available to inspect")

## 2. Process and Transform Data

Now we'll process the NRI data to create our wide format CSV with County FIPS codes and hazard risk scores.

In [ ]:
def process_nri_data(df):
    """
    Process NRI data into wide format with FIPS codes and hazard scores.
    
    Args:
        df: Raw NRI DataFrame
    
    Returns:
        Processed DataFrame in wide format
    """
    if df.empty:
        print("Cannot process empty DataFrame")
        return pd.DataFrame()
    
    print("Processing NRI data...")
    
    # Identify FIPS code column (may vary by API version)
    fips_cols = [col for col in df.columns if 'fips' in col.lower() or 'stcofips' in col.lower()]
    if not fips_cols:
        print("Warning: No FIPS code column found")
        fips_col = df.columns[0]  # Use first column as fallback
    else:
        fips_col = fips_cols[0]
    
    print(f"Using {fips_col} as FIPS code column")
    
    # Create base dataframe with FIPS codes
    result_df = df[[fips_col]].copy()
    result_df.rename(columns={fips_col: 'county_fips'}, inplace=True)
    
    # Add state and county names if available
    name_cols = [col for col in df.columns if any(x in col.lower() for x in ['state', 'county', 'name'])]
    for col in name_cols[:3]:  # Add up to 3 name columns
        if col in df.columns:
            result_df[col.lower()] = df[col]
    
    # Add hazard risk scores
    for hazard_code, hazard_name in HAZARD_MAPPING.items():
        # Look for risk score columns matching this hazard
        # FEMA NRI uses patterns like RISK_RATNG (risk rating), RISK_SCORE, etc.
        hazard_cols = [col for col in df.columns if hazard_code.lower() in col.lower()]
        
        if hazard_cols:
            # Prefer risk score columns
            score_cols = [col for col in hazard_cols if 'score' in col.lower() or 'ratng' in col.lower()]
            if score_cols:
                result_df[f'{hazard_code}_risk_score'] = df[score_cols[0]]
            else:
                result_df[f'{hazard_code}_risk_score'] = df[hazard_cols[0]]
    
    # Add overall risk score if available
    risk_cols = [col for col in df.columns if 'risk' in col.lower() and 'score' in col.lower() and not any(h in col.lower() for h in HAZARD_MAPPING.keys())]
    if risk_cols:
        result_df['overall_risk_score'] = df[risk_cols[0]]
    
    print(f"Processed data shape: {result_df.shape}")
    return result_df

# Process the data
processed_df = process_nri_data(nri_df)

In [ ]:
# Inspect processed data
if not processed_df.empty:
    print(f"Processed data shape: {processed_df.shape}")
    print(f"\nColumns: {list(processed_df.columns)}")
    print(f"\nFirst few rows:")
    display(processed_df.head(10))
    
    # Show summary statistics for risk scores
    risk_cols = [col for col in processed_df.columns if 'risk_score' in col]
    if risk_cols:
        print(f"\nRisk score summary statistics:")
        display(processed_df[risk_cols].describe())
else:
    print("No processed data available")

## 3. Canada and Mexico Extension (Future)

This section is a placeholder for extending the analysis to Canada and Mexico.
Additional data sources would be needed:

- **Canada**: Public Safety Canada's Risk Profile, Statistics Canada
- **Mexico**: CENAPRED (Centro Nacional de Prevención de Desastres)

For now, we'll focus on US data from FEMA NRI.

In [ ]:
# Placeholder for international data integration
def fetch_canada_risk_data():
    """
    Placeholder for fetching Canadian disaster risk data.
    Would integrate with Public Safety Canada or other sources.
    """
    print("Canada data integration: Not yet implemented")
    return pd.DataFrame()

def fetch_mexico_risk_data():
    """
    Placeholder for fetching Mexico disaster risk data.
    Would integrate with CENAPRED or other sources.
    """
    print("Mexico data integration: Not yet implemented")
    return pd.DataFrame()

# Note: These would be integrated in future versions
# canada_df = fetch_canada_risk_data()
# mexico_df = fetch_mexico_risk_data()

## 4. Export Per-State CSVs

Export the processed data to per-state CSV files. This approach:
- Reduces data lookup costs (smaller files per state)
- Makes state-level analysis easier
- Improves performance when loading specific states
- Allows parallel processing of states

In [ ]:
def export_per_state_csvs(df, output_dir):
    """
    Export DataFrame to per-state CSV files in wide format.
    
    Args:
        df: DataFrame to export (must have 'state' or similar column)
        output_dir: Directory to save CSV files
    
    Returns:
        Dictionary mapping state names to file paths
    """
    if df.empty:
        print("Cannot export empty DataFrame")
        return {}
    
    # Ensure output directory exists
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Find state column (may vary by data source)
    state_col = None
    for col in df.columns:
        if 'state' in col.lower() and 'fips' not in col.lower():
            state_col = col
            break
    
    if not state_col:
        print("Warning: No state column found. Exporting as single file.")
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_file = output_dir / f'all_states_risk_data_{timestamp}.csv'
        df.to_csv(output_file, index=False)
        print(f"Exported {len(df):,} records to: {output_file}")
        return {'all_states': output_file}
    
    print(f"Using '{state_col}' column for state grouping\n")
    
    # Group by state and export
    exported_files = {}
    states = sorted(df[state_col].unique())
    
    print(f"Exporting data for {len(states)} states...")
    
    for state in tqdm(states, desc="Exporting per-state CSVs"):
        # Filter data for this state
        state_df = df[df[state_col] == state].copy()
        
        # Create filename (sanitize state name)
        state_name = str(state).lower().replace(' ', '_').replace('-', '_')
        state_name = ''.join(c for c in state_name if c.isalnum() or c == '_')
        
        timestamp = datetime.now().strftime('%Y%m%d')
        output_file = output_dir / f'{state_name}_risk_data_{timestamp}.csv'
        
        # Export to CSV
        state_df.to_csv(output_file, index=False)
        
        exported_files[state] = output_file
    
    print(f"\n✓ Exported {len(exported_files)} state files to: {output_dir.absolute()}")
    
    # Print summary statistics
    total_size = sum(f.stat().st_size for f in exported_files.values())
    print(f"Total size: {total_size / 1024:.2f} KB")
    print(f"Average per state: {total_size / len(exported_files) / 1024:.2f} KB")
    
    # Show sample of exported files
    print(f"\nSample exported files:")
    for state, filepath in list(exported_files.items())[:5]:
        file_size = filepath.stat().st_size / 1024
        state_records = len(df[df[state_col] == state])
        print(f"  {filepath.name} ({state_records} counties, {file_size:.1f} KB)")
    
    if len(exported_files) > 5:
        print(f"  ... and {len(exported_files) - 5} more")
    
    return exported_files

# Export the data per-state
if not processed_df.empty:
    exported_files = export_per_state_csvs(processed_df, OUTPUT_DIR)
else:
    print("No data to export")

## 5. Data Visualization

Create some basic visualizations to understand the risk distribution.

In [ ]:
# Visualize risk score distributions
if not processed_df.empty:
    risk_cols = [col for col in processed_df.columns if 'risk_score' in col]
    
    if risk_cols:
        # Create figure with subplots
        n_cols = min(3, len(risk_cols))
        n_rows = (len(risk_cols) + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
        if n_rows == 1 and n_cols == 1:
            axes = [axes]
        else:
            axes = axes.flatten() if n_rows > 1 else axes
        
        for idx, col in enumerate(risk_cols[:len(axes)]):
            # Convert to numeric, handling any non-numeric values
            data = pd.to_numeric(processed_df[col], errors='coerce').dropna()
            
            if len(data) > 0:
                axes[idx].hist(data, bins=50, edgecolor='black', alpha=0.7)
                axes[idx].set_title(col.replace('_', ' ').title())
                axes[idx].set_xlabel('Risk Score')
                axes[idx].set_ylabel('Frequency')
                axes[idx].grid(True, alpha=0.3)
        
        # Hide empty subplots
        for idx in range(len(risk_cols), len(axes)):
            axes[idx].set_visible(False)
        
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f'risk_distributions_{timestamp}.png', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\nSaved visualization to: {OUTPUT_DIR / f'risk_distributions_{timestamp}.png'}")
    else:
        print("No risk score columns found for visualization")
else:
    print("No data available for visualization")

## 6. Summary and Next Steps

This notebook provides a foundation for analyzing disaster risk projection data. 

### Current Capabilities:
- Fetches live FEMA National Risk Index data for US counties (bypasses CORS via Python requests)
- Processes data into wide format with County FIPS codes
- Maps hazards to RAPT hazard codes (18 types)
- **Exports per-state CSV files** for reduced data lookup costs
- Provides basic visualizations
- Falls back to sample data if API unavailable

### Per-State CSV Benefits:
- **Faster lookups**: Smaller file sizes per state
- **Easier state analysis**: Focus on specific states without loading all data
- **Better performance**: Load only needed states
- **Parallel processing**: Process multiple states simultaneously
- **Version control friendly**: Smaller diffs when data updates

### CORS Handling:
- **Python requests bypass CORS**: Running in Jupyter notebooks, Python's `requests` library makes direct HTTP calls that are not subject to browser CORS restrictions
- **Alternative**: If API access is blocked, download the full NRI CSV from FEMA's website
- **No browser issues**: This notebook doesn't run in browsers, so CORS is not a concern

### Future Enhancements:
1. **International Data**: Integrate Canadian and Mexican disaster risk data
2. **GFDRR CCDR Tools**: Incorporate global climate and disaster risk data
3. **EM-DAT Integration**: Add historical disaster event data
4. **Advanced Mapping**: Create interactive maps with Folium or Plotly
5. **Time Series Analysis**: Track risk changes over time
6. **Climate Projections**: Incorporate future climate scenario modeling
7. **API Caching**: Cache API responses to reduce repeated calls

### Resources:
- [FEMA National Risk Index](https://www.fema.gov/about/openfema/data-sets/national-risk-index-data)
- [FEMA OpenFEMA API](https://www.fema.gov/about/openfema/api)
- [RAPT Tool](https://www.fema.gov/emergency-managers/practitioners/resilience-analysis-and-planning-tool)
- [Risk Data Library](https://docs.riskdatalibrary.org/)
- [GFDRR CCDR Tools](https://github.com/GFDRR/CCDR-tools)

In [ ]:
# Print final summary
print("="*80)
print("DISASTER RISK ANALYSIS SUMMARY")
print("="*80)
print(f"\nTimestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

if not processed_df.empty:
    print(f"\nData processed: {len(processed_df):,} counties")
    
    # Count states
    state_col = None
    for col in processed_df.columns:
        if 'state' in col.lower() and 'fips' not in col.lower():
            state_col = col
            break
    
    if state_col:
        num_states = processed_df[state_col].nunique()
        print(f"States covered: {num_states}")
        print(f"Average counties per state: {len(processed_df) / num_states:.1f}")
else:
    print("\nNo data processed")

print(f"\nHazard types analyzed: {len(HAZARD_MAPPING)}")
print(f"\nOutput format: Per-state CSV files")
print(f"Output directory: {OUTPUT_DIR.absolute()}")

if 'exported_files' in globals() and exported_files:
    print(f"\nFiles exported: {len(exported_files)} state files")

print("\n" + "="*80)